In [0]:
%run ../config/config-env


In [0]:
from pyspark.sql import functions as F

In [0]:
constructor_df = f'{catalog_name}.{silver_schema}.constructors'
ref_region_df = f'{catalog_name}.{gold_schema}.ref_nationality_region'
target_table = f'{catalog_name}.{gold_schema}.gold_constructors_dim'


In [0]:
constructor_df_read = spark.read.table(constructor_df)
ref_region_read_df = spark.read.table(ref_region_df)

In [0]:
joined_df =( 
            constructor_df_read
                .join(
                    ref_region_read_df, 
                    constructor_df_read.nationality == ref_region_read_df.nationality,
                    'left'
                    )
            ).select(
                constructor_df_read.constructor_Id,
                constructor_df_read.constructor_name,
                ref_region_read_df.nationality,
                ref_region_read_df.region
            )

In [0]:
(
    joined_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))